# Subclass by Frame (Cell + Nucleus) - vem 1uM, 48hrs Pre-Treatment

For the `vem_1uM_48hrsbefore` dataset: frames 1-2 are *before* drug
addition (pre-treatment baseline), frame 3 is day 0 (when 1uM
vemurafenib was added). Every available cell/nucleus across these 3
frames went through CAJAL together (`main.py`), producing one shared
UMAP embedding.

This notebook plots each frame's cells on that shared embedding,
colored by subclass (dying, persisting, slow proliferator, fast
proliferator) via KDE contours - same convention as
`naive_vs_vem/notebooks/03_vem_subclasses_over_time.ipynb`. Unlike that
analysis, there's no condition comparison here (a single dataset), so
each page shows only that frame's own cells, restricted to the 4
subclasses.

Requires `main.py` to have been run first (produces
`selected_metadata.csv`, `cell_umap.npz`, `nuc_umap.npz` under
`output/cajal_output/`).

In [ ]:
import os
import sys

import matplotlib.pyplot as plt

# assumes Jupyter's cwd is this notebook's own directory
# (analyses/vem_48hr_pretreatment/notebooks/), so the analysis dir - and
# config.py - is one level up, and the repo root - where shared/ lives -
# is two more levels up from there
NOTEBOOK_DIR = os.getcwd()
ANALYSIS_DIR = os.path.abspath(os.path.join(NOTEBOOK_DIR, '..'))
REPO_ROOT    = os.path.abspath(os.path.join(ANALYSIS_DIR, '..', '..'))
sys.path.insert(0, REPO_ROOT)
sys.path.insert(0, ANALYSIS_DIR)
import config

# config.py's OUTPUT_DIR/CAJAL_OUTPUT_DIR are relative to the analysis
# directory, not this notebook's own cwd - re-anchor them to an absolute
# path once, here
config.OUTPUT_DIR       = os.path.join(ANALYSIS_DIR, config.OUTPUT_DIR)
config.CAJAL_OUTPUT_DIR = os.path.join(ANALYSIS_DIR, config.CAJAL_OUTPUT_DIR)

from shared.load_data import load_full_dataset
from shared.density_plots import export_kde_by_frame

## Plot: UMAP with per-subclass kernel density, one page per frame

One combined, multi-page PDF (one page per frame: 1, 2, 3) and one SVG
per frame, written to `output/plots/01_subclass_by_frame/pdfs/` and
`.../svgs/`.

In [ ]:
SUBCLASS_COLORS = {
    'dying':       'tab:red',
    'persisting':  'tab:purple',
    'slow_prolif': 'tab:orange',
    'fast_prolif': 'tab:green',
}
NOTEBOOK_PLOTS_DIR = os.path.join(config.OUTPUT_DIR, 'plots', '01_subclass_by_frame')

def process_modality(modality):
    '''
    Load this modality's embedding + metadata (frames 1-3, every
    subclass) and export one PDF/SVG pair per frame with a KDE contour
    per subclass.

    Inputs:
        modality: 'cell' or 'nuc'
    Outputs:
        df: the full merged dataset for this modality (in-memory only)
    '''
    df = load_full_dataset(config.CAJAL_OUTPUT_DIR, modality)
    print(f"[{modality}] {len(df)} cells across {df['frame'].nunique()} frames, "
          f"{df['subclass'].nunique()} subclasses")

    export_kde_by_frame(df, 'subclass', SUBCLASS_COLORS, NOTEBOOK_PLOTS_DIR,
                         modality, title_prefix=f'{modality} - subclass by frame')

    return df

## Run: cell morphology

In [ ]:
cell_df = process_modality('cell')
cell_df.head()

## Run: nuclear morphology

In [ ]:
nuc_df = process_modality('nuc')
nuc_df.head()